# 04. Master Manifest 구성

검증을 마친 FMA 원곡 296개와 Echoes의 TTA 생성물 3,162개를 하나의 manifest로 합친다. REAL과 FAKE가 같은 `original_audio` 집합을 공유하는지 확인하고, 파일 누락과 그룹 구조를 점검한 뒤 `master_manifest.csv`로 저장한다.

이후 데이터 분할에서는 같은 원곡과 해당 원곡에서 파생된 생성물이 하나의 split에만 들어가도록 `original_audio`를 그룹 변수로 사용한다.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

ECHOES_ROOT = PROJECT_ROOT / "data/raw/Echoes/Echoes"
ECHOES_MANIFEST = ECHOES_ROOT / "dataset_manifest.csv"

FMA_MAPPING = PROJECT_ROOT / "data/metadata/fma_real_mapping.csv"
FMA_AUDIO_DIR = PROJECT_ROOT / "data/raw/FMA/selected_30s"

OUTPUT_PATH = PROJECT_ROOT / "data/metadata/master_manifest.csv"

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("ECHOES_MANIFEST  :", ECHOES_MANIFEST)
print("FMA_MAPPING      :", FMA_MAPPING)
print("OUTPUT_PATH      :", OUTPUT_PATH)


PROJECT_ROOT     : <PROJECT_ROOT>
ECHOES_MANIFEST  : <PROJECT_ROOT>/data/raw/Echoes/Echoes/dataset_manifest.csv
FMA_MAPPING      : <PROJECT_ROOT>/data/metadata/fma_real_mapping.csv
OUTPUT_PATH      : <PROJECT_ROOT>/data/metadata/master_manifest.csv


**실행 결과.** 프로젝트 루트와 Echoes manifest, FMA 매핑, 결과 CSV의 경로를 설정했다. 통합 결과는 `data/metadata/master_manifest.csv`에 저장하도록 지정했다.


## 1. Echoes TTA 데이터 정리

Echoes manifest에서 `type == "TTA"`인 행을 추출한다. 같은 `path_in_dataset`이 여러 원곡에 연결된 경우에는 어느 원곡에 속하는지 확정할 수 없으므로 해당 행을 모두 제외한다.


In [3]:
echoes = pd.read_csv(ECHOES_MANIFEST)

print("Echoes manifest rows:", len(echoes))
print("Columns:", echoes.columns.tolist())

tta = echoes[echoes["type"] == "TTA"].copy()

print("\nTTA rows:", len(tta))
print("TTA unique original_audio:", tta["original_audio"].nunique())


Echoes manifest rows: 4468
Columns: ['path_in_dataset', 'original_audio', 'generator', 'type', 'genre', 'description', 'duration']

TTA rows: 3165
TTA unique original_audio: 296


**실행 결과.** Echoes manifest는 4,468행이며, 이 중 TTA는 3,165행이다. TTA 데이터에는 서로 다른 `original_audio` 296개가 포함되어 있다.


In [4]:
dup_path_mask = tta["path_in_dataset"].duplicated(keep=False)
duplicated_tta = tta[dup_path_mask].copy()

print("Duplicated TTA rows        :", len(duplicated_tta))
print("Duplicated TTA unique paths:", duplicated_tta["path_in_dataset"].nunique())

if len(duplicated_tta):
    display(
        duplicated_tta[
            ["path_in_dataset", "original_audio", "generator", "genre"]
        ].sort_values("path_in_dataset")
    )


Duplicated TTA rows        : 3
Duplicated TTA unique paths: 1


,path_in_dataset,original_audio,generator,genre
4454,TTA/musicgen/_musicgen_TTA_001.wav,В Наших Сердцах - Чокнутый Пропеллер,musicgen,Rock
4456,TTA/musicgen/_musicgen_TTA_001.wav,Глазами Детей - Чокнутый Пропеллер,musicgen,Rock
4458,TTA/musicgen/_musicgen_TTA_001.wav,Кортни Лав - Чокнутый Пропеллер,musicgen,Rock


**실행 결과.** 중복된 TTA 행은 3개였고 모두 `TTA/musicgen/_musicgen_TTA_001.wav` 한 경로를 공유했다. 이 경로가 서로 다른 원곡 3개에 연결되어 있어 세 행 모두 정제 대상에 포함했다.


In [5]:
tta_clean = tta[~dup_path_mask].copy().reset_index(drop=True)

print("===== CLEAN TTA =====")
print("Rows                 :", len(tta_clean))
print("Unique original_audio:", tta_clean["original_audio"].nunique())
print("Generators           :", tta_clean["generator"].nunique())

print("\nGenre distribution:")
print(tta_clean["genre"].value_counts())


===== CLEAN TTA =====
Rows                 : 3162
Unique original_audio: 296
Generators           : 12

Genre distribution:
genre
Electronic    1131
Rock          1127
Pop            904
Name: count, dtype: int64


**실행 결과.** 중복 3행을 제외한 TTA 데이터는 3,162행, 원곡 296개, 생성기 12개다. 장르별로는 Electronic 1,131개, Rock 1,127개, Pop 904개였다.


## 2. FMA REAL 매핑 확인

앞 단계에서 품질 검사를 마친 FMA 매핑을 불러오고 `track_id`를 정수형으로 맞춘다. 각 원곡이 서로 다른 FMA 트랙 하나와 연결되는지도 함께 확인한다.


In [6]:
real_mapping = pd.read_csv(FMA_MAPPING)
real_mapping["track_id"] = real_mapping["track_id"].astype(int)

print("===== FMA REAL =====")
print("Rows                 :", len(real_mapping))
print("Unique original_audio:", real_mapping["original_audio"].nunique())
print("Unique track_id      :", real_mapping["track_id"].nunique())

display(real_mapping.head())


===== FMA REAL =====
Rows                 : 296
Unique original_audio: 296
Unique track_id      : 296


,original_audio,genre,track_id,title,artist,genre_top,license,duration,subset,candidate_count,license_allowed,license_fallback,genre_match
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,140932,"10,000 People Chanting, ""I'm an Individual""",Nihilore,Electronic,Creative Commons Attribution,372,medium,1,True,False,True
1,1984 - Punk Rock Opera,Rock,149410,1984,Punk Rock Opera,Rock,Attribution,200,medium,2,True,False,True
2,2 (Wasn't There) - Isle of Pine,Rock,66449,2 (Wasn't There),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,108,medium,1,True,False,True
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,114244,2Much (Andy Spinelli & Alex Sánchez House Edit),Tentacles,Electronic,Attribution,486,medium,1,True,False,True
4,3 am West End - statusq,Electronic,112378,3 am West End,statusq,Electronic,Attribution,291,medium,1,True,False,True


**실행 결과.** FMA 매핑은 296행이며 `original_audio`와 `track_id`도 각각 296개로 모두 고유했다. 따라서 각 원곡이 FMA 트랙 하나와 일대일로 연결된다.


## 3. REAL과 FAKE의 원곡 집합 비교

REAL과 FAKE의 `original_audio` 집합이 같은지 비교한다. 두 집합이 같아야 이후 원곡 단위 분할에서 각 원곡의 REAL과 FAKE를 함께 배정할 수 있다.


In [7]:
real_groups = set(real_mapping["original_audio"])
fake_groups = set(tta_clean["original_audio"])

only_real = sorted(real_groups - fake_groups)
only_fake = sorted(fake_groups - real_groups)

print("REAL groups:", len(real_groups))
print("FAKE groups:", len(fake_groups))
print("Only REAL  :", len(only_real))
print("Only FAKE  :", len(only_fake))

if only_real:
    print("\nOnly REAL examples:", only_real[:10])

if only_fake:
    print("\nOnly FAKE examples:", only_fake[:10])

assert real_groups == fake_groups, (
    "REAL과 FAKE의 original_audio 집합이 일치하지 않습니다."
)

print("\noriginal_audio group match: PASS")


REAL groups: 296
FAKE groups: 296
Only REAL  : 0
Only FAKE  : 0

original_audio group match: PASS


**실행 결과.** REAL과 FAKE에 포함된 원곡은 각각 296개였고, 한쪽에만 있는 원곡은 없었다. 두 `original_audio` 집합이 완전히 일치해 assertion을 통과했다.


## 4. REAL manifest 생성

FMA 매핑에 `REAL` 라벨과 FMA 기준 상대 경로를 추가한다. REAL 데이터에는 생성기 정보가 없으므로 `generator`는 결측값으로 둔다.


In [15]:
real_manifest = real_mapping.copy()

real_manifest["label"] = "REAL"
real_manifest["label_id"] = 0
real_manifest["source"] = "FMA"
real_manifest["generator"] = pd.NA
real_manifest["description"] = ""

real_manifest["audio_path"] = real_manifest["track_id"].apply(
    lambda tid: str(
        Path("data/raw/FMA/selected_30s")
        / f"{int(tid) // 1000:03d}"
        / f"{int(tid):06d}.mp3"
    )
)

real_manifest["path_in_dataset"] = pd.NA

display(
    real_manifest[
        [
            "original_audio", "label", "genre", "generator",
            "track_id", "audio_path"
        ]
    ].head()
)


,original_audio,label,genre,generator,track_id,audio_path
0,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,Electronic,<NA>,140932,data/raw/FMA/selected_30s/140/140932.mp3
1,1984 - Punk Rock Opera,REAL,Rock,<NA>,149410,data/raw/FMA/selected_30s/149/149410.mp3
2,2 (Wasn't There) - Isle of Pine,REAL,Rock,<NA>,66449,data/raw/FMA/selected_30s/066/066449.mp3
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,Electronic,<NA>,114244,data/raw/FMA/selected_30s/114/114244.mp3
4,3 am West End - statusq,REAL,Electronic,<NA>,112378,data/raw/FMA/selected_30s/112/112378.mp3


**실행 결과.** REAL 행에 `label_id=0`, `source=FMA`를 지정하고 `track_id`로 오디오 상대 경로를 만들었다. 출력 예시에서 140932번 트랙은 `data/raw/FMA/selected_30s/140/140932.mp3`로 연결되었다.


## 5. FAKE manifest 생성

정제한 TTA 행에 `FAKE` 라벨을 부여하고 Echoes 루트를 기준으로 오디오 상대 경로를 만든다. 생성기와 설명 정보는 원본 manifest의 값을 유지한다.


In [16]:
fake_manifest = tta_clean.copy()

fake_manifest["label"] = "FAKE"
fake_manifest["label_id"] = 1
fake_manifest["source"] = "Echoes"
fake_manifest["track_id"] = pd.NA

fake_manifest["audio_path"] = fake_manifest["path_in_dataset"].apply(
    lambda p: str(Path("data/raw/Echoes/Echoes") / str(p))
)

display(
    fake_manifest[
        [
            "original_audio", "label", "genre", "generator",
            "description", "audio_path"
        ]
    ].head()
)


,original_audio,label,genre,generator,description,audio_path
0,"10,000 People Chanting, ""I'm an Individual"" - ...",FAKE,Electronic,acestep,"cinematic, idm, downtempo, layered, swelling, ...",data/raw/Echoes/Echoes/TTA/acestep/10000_Peopl...
1,1984 - Punk Rock Opera,FAKE,Rock,acestep,"hardcore-punk, political, d-beat, shouted-chor...",data/raw/Echoes/Echoes/TTA/acestep/1984_Punk_R...
2,2Much (Andy Spinelli & Alex Sánchez House Edit...,FAKE,Electronic,acestep,"house, four-on-the-floor, piano-stabs, filtere...",data/raw/Echoes/Echoes/TTA/acestep/2Much_Andy_...
3,2 (Wasn't There) - Isle of Pine,FAKE,Rock,acestep,"post-rock, melancholic, clean-guitars, spaciou...",data/raw/Echoes/Echoes/TTA/acestep/2_Wasnt_The...
4,3 am West End - statusq,FAKE,Electronic,acestep,"chillhop, nocturnal, lofi, warm, headnod, mell...",data/raw/Echoes/Echoes/TTA/acestep/3_am_West_E...


**실행 결과.** 정제된 TTA 행에 `label_id=1`, `source=Echoes`를 지정했다. 출력 예시에서 생성기와 설명은 유지되었고, `audio_path`는 Echoes 데이터 루트부터 시작하는 상대 경로로 변환되었다.


## 6. REAL과 FAKE 통합

두 manifest의 열 순서를 맞춘 뒤 행 방향으로 결합하고, 각 행에 중복 없는 `sample_id`를 부여한다.


In [17]:
master_columns = [
    "original_audio",
    "label",
    "label_id",
    "source",
    "genre",
    "generator",
    "audio_path",
    "track_id",
    "description",
    "path_in_dataset",
]

real_part = real_manifest[master_columns].copy()
fake_part = fake_manifest[master_columns].copy()

master_manifest = pd.concat(
    [real_part, fake_part],
    ignore_index=True,
)

master_manifest.insert(
    0,
    "sample_id",
    [f"sample_{i:05d}" for i in range(len(master_manifest))],
)

print("===== MASTER MANIFEST =====")
print("Rows                 :", len(master_manifest))
print("Unique original_audio:", master_manifest["original_audio"].nunique())

print("\nLabel distribution:")
print(master_manifest["label"].value_counts())

display(master_manifest.head())


===== MASTER MANIFEST =====
Rows                 : 3458
Unique original_audio: 296

Label distribution:
label
FAKE    3162
REAL     296
Name: count, dtype: int64


,sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,description,path_in_dataset
0,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,<NA>,data/raw/FMA/selected_30s/140/140932.mp3,140932,,<NA>
1,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,<NA>,data/raw/FMA/selected_30s/149/149410.mp3,149410,,<NA>
2,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,<NA>,data/raw/FMA/selected_30s/066/066449.mp3,66449,,<NA>
3,sample_00003,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,0,FMA,Electronic,<NA>,data/raw/FMA/selected_30s/114/114244.mp3,114244,,<NA>
4,sample_00004,3 am West End - statusq,REAL,0,FMA,Electronic,<NA>,data/raw/FMA/selected_30s/112/112378.mp3,112378,,<NA>


**실행 결과.** 통합 manifest는 3,458행과 원곡 296개로 구성되었다. 라벨별 행 수는 FAKE 3,162개, REAL 296개이며 `sample_id`는 `sample_00000`부터 순서대로 부여되었다.


## 7. 원곡별 구성 확인

`original_audio`별로 REAL 수, FAKE 수, 장르 수, 생성기 수를 집계한다. 각 그룹에는 REAL이 정확히 1개 있어야 하며 장르도 하나로 일치해야 한다.


In [18]:
group_check = (
    master_manifest
    .groupby("original_audio")
    .agg(
        total_samples=("sample_id", "size"),
        real_count=("label", lambda x: (x == "REAL").sum()),
        fake_count=("label", lambda x: (x == "FAKE").sum()),
        genre_count=("genre", "nunique"),
        generator_count=("generator", "nunique"),
    )
    .reset_index()
)

bad_real_count = group_check[group_check["real_count"] != 1]
bad_genre_count = group_check[group_check["genre_count"] != 1]

print("Groups                :", len(group_check))
print("REAL count != 1       :", len(bad_real_count))
print("Genre count != 1      :", len(bad_genre_count))

display(group_check.head())


Groups                : 296
REAL count != 1       : 0
Genre count != 1      : 0


,original_audio,total_samples,real_count,fake_count,genre_count,generator_count
0,"10,000 People Chanting, ""I'm an Individual"" - ...",6,1,5,1,5
1,1984 - Punk Rock Opera,14,1,13,1,12
2,2 (Wasn't There) - Isle of Pine,17,1,16,1,12
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,6,1,5,1,5
4,3 am West End - statusq,6,1,5,1,5


**실행 결과.** 296개 원곡 그룹을 검사한 결과 REAL 수가 1이 아닌 그룹과 장르가 둘 이상인 그룹은 모두 0개였다. 원곡별 FAKE 수는 서로 달랐지만 각 그룹의 REAL 1개 조건은 유지되었다.


## 8. 오디오 파일 존재 여부 확인

manifest의 상대 경로를 프로젝트 루트와 결합해 모든 오디오 파일이 실제로 존재하는지 검사한다.


In [19]:
master_manifest["file_exists"] = master_manifest["audio_path"].apply(
    lambda p: (PROJECT_ROOT / p).exists()
)

missing_files = master_manifest[~master_manifest["file_exists"]].copy()

print("Total rows   :", len(master_manifest))
print("Files exist :", int(master_manifest["file_exists"].sum()))
print("Missing     :", len(missing_files))

if len(missing_files):
    display(
        missing_files[
            ["sample_id", "original_audio", "label", "audio_path"]
        ].head(30)
    )


Total rows   : 3458
Files exist : 3458
Missing     : 0


**실행 결과.** manifest의 3,458개 경로가 모두 실제 파일과 연결되었으며 누락 파일은 0개였다.


## 9. 최종 품질 검사

전체 행 수와 라벨 수, 원곡별 구성, 파일 누락, `sample_id` 중복을 한 번에 확인한다.


In [20]:
qc_summary = pd.DataFrame({
    "check": [
        "master_rows",
        "unique_original_audio",
        "real_rows",
        "fake_rows",
        "groups_with_real_count_not_1",
        "groups_with_genre_count_not_1",
        "missing_audio_files",
        "duplicate_sample_id",
    ],
    "value": [
        len(master_manifest),
        master_manifest["original_audio"].nunique(),
        int((master_manifest["label"] == "REAL").sum()),
        int((master_manifest["label"] == "FAKE").sum()),
        len(bad_real_count),
        len(bad_genre_count),
        len(missing_files),
        int(master_manifest["sample_id"].duplicated().sum()),
    ],
})

display(qc_summary)

core_qc_pass = (
    len(master_manifest) == 3458
    and master_manifest["original_audio"].nunique() == 296
    and int((master_manifest["label"] == "REAL").sum()) == 296
    and int((master_manifest["label"] == "FAKE").sum()) == 3162
    and len(bad_real_count) == 0
    and len(bad_genre_count) == 0
    and len(missing_files) == 0
    and int(master_manifest["sample_id"].duplicated().sum()) == 0
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", core_qc_pass)


,check,value
0,master_rows,3458
1,unique_original_audio,296
2,real_rows,296
3,fake_rows,3162
4,groups_with_real_count_not_1,0
5,groups_with_genre_count_not_1,0
6,missing_audio_files,0
7,duplicate_sample_id,0


===== FINAL RESULT =====
Core QC PASS: True


**실행 결과.** 전체 3,458행, 원곡 296개, REAL 296개, FAKE 3,162개를 확인했다. 그룹 구성 오류, 장르 불일치, 파일 누락, `sample_id` 중복은 모두 0개로 `Core QC PASS`가 `True`였다.


## 10. 결과 저장

모든 검사 조건을 만족한 경우에만 통합 manifest를 CSV로 저장한다.


In [21]:
if not core_qc_pass:
    raise RuntimeError(
        "Core QC가 통과하지 않았습니다. 저장 전에 위 결과를 확인하세요."
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

master_manifest.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", OUTPUT_PATH)
print("Rows :", len(master_manifest))


Saved: <PROJECT_ROOT>/data/metadata/master_manifest.csv
Rows : 3458


**실행 결과.** 품질 검사를 통과한 3,458행을 `data/metadata/master_manifest.csv`에 저장했다.


## 11. 정리

FMA REAL 296개와 Echoes FAKE 3,162개를 결합해 3,458행의 manifest를 만들었다. 296개 원곡 그룹에는 REAL이 하나씩 포함되어 있고, 그룹 안의 장르도 일치했다. 연결된 오디오 파일 3,458개는 모두 존재했으며 `sample_id` 중복도 없었다.


## 다음 단계

`master_manifest.csv`의 296개 `original_audio` 그룹을 Train, Validation, Test로 나눈다. 같은 원곡에 속하는 REAL과 FAKE는 모두 같은 split에 배정한다.
